Проверка в Postman (по одному запросу)

Авторизация (один раз на запрос или на коллекцию): вкладка Authorization → Type Basic Auth → Username = amyakaaaaadm@gmail.com, Password = a5dc273b-5151-465e-aef5-057a9b10dbc9.
Запрос 1 — статусы (чтобы узнать, как называются «закрытые»): GET https://ВАШ_ДОМЕН/api/v2/status_list/. Запомни id закрытого/решённого статуса.
Запрос 2 — тикеты: GET https://ВАШ_ДОМЕН/api/v2/tickets/?page=1. Возьми любой id из ответа.
Запрос 3 — ответы тикета: GET https://ВАШ_ДОМЕН/api/v2/tickets/<id>/posts/. Увидишь посты с user_id и text.
Когда запрос работает — нажми справа кнопку </> (Code) → выбери Python - requests → скопируй сгенерированный код. Это и есть «взять код».

Следи за заголовком X-Rate-Limit-Remaining в ответах: при превышении лимита доступ блокируется на 20 минут, так что не гоняй запросы слишком часто.

Как влить это в обучение бота. У тебя одобренные ответы лежат в approved.csv и оттуда индексируются в ChromaDB. Отсюда два пути:

Простой: привести daodesk_cases.csv к формату approved.csv (те же колонки, плюс категория, если она там есть) и переиндексировать — твоя существующая логика подхватит новые пары.
Прямой: в цикле прогонять каждую пару через embedding-модель и добавлять в ту же коллекцию ChromaDB — как review_manager делает для одного одобренного ответа, только пачкой.

Чтобы я дописал прямую интеграцию точно под твой код — скинь app/learning/review_manager.py и пример строки из approved.csv (с заголовком). Тогда сделаю так, чтобы импорт сразу попадал в базу знаний без ручного маппинга.

"""
Импорт реальных кейсов из DaoDesk в обучающую базу бота.

Что делает:
  1. Берёт закрытые тикеты (постранично).
  2. Для каждого тикета вытаскивает посты (ответы).
  3. Разделяет сообщения клиента (вопрос) и сотрудников (ответ) по user_id.
  4. Чистит HTML и сохраняет пары "вопрос — ответ" в CSV.

Запуск:
  pip install requests beautifulsoup4
  python import_daodesk_cases.py
"""

import csv
import time

import requests
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# ─────────────────────────── НАСТРОЙКИ ───────────────────────────
DOMAIN = "https://ВАШ_ДОМЕН"            # ← адрес твоего DaoDesk, например https://company.daodesk.ai
EMAIL = "amyakaaaaadm@gmail.com"
API_KEY = "a5dc273b-5151-465e-aef5-057a9b10dbc9"

# id статусов "закрытый/решённый" — узнай через GET /status_list/ (в Postman).
# Если оставить пустую строку "" — возьмутся ВСЕ тикеты (в т.ч. без ответов).
CLOSED_STATUSES = "closed,resolved"

OUTPUT_CSV = "daodesk_cases.csv"
REQUEST_PAUSE = 0.4                      # пауза между запросами, чтобы не упереться в rate limit
# ──────────────────────────────────────────────────────────────────

BASE_URL = f"{DOMAIN}/api/v2"
auth = HTTPBasicAuth(EMAIL, API_KEY)


def strip_html(html: str) -> str:
    """Убирает HTML-теги, оставляет чистый текст."""
    return BeautifulSoup(html or "", "html.parser").get_text(separator=" ", strip=True)


def get_tickets(page: int) -> dict:
    params = {"page": page}
    if CLOSED_STATUSES:
        params["status_list"] = CLOSED_STATUSES
    resp = requests.get(f"{BASE_URL}/tickets/", params=params, auth=auth, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_posts(ticket_id: int) -> list:
    resp = requests.get(
        f"{BASE_URL}/tickets/{ticket_id}/posts/",
        params={"page": 1},
        auth=auth,
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json().get("data", [])


def build_qa(ticket: dict) -> dict | None:
    """Собирает пару вопрос-ответ из одного тикета."""
    ticket_id = ticket["id"]
    client_id = ticket.get("user_id")

    posts = get_posts(ticket_id)
    if not posts:
        return None

    # сортируем по id (по возрастанию ≈ по времени создания)
    posts.sort(key=lambda p: p.get("id", 0))

    questions = [strip_html(p.get("text", "")) for p in posts if p.get("user_id") == client_id]
    answers = [strip_html(p.get("text", "")) for p in posts if p.get("user_id") != client_id]

    # вопрос = тема тикета + сообщения клиента; ответ = ответы сотрудников
    question = (ticket.get("title", "") + " " + " ".join(q for q in questions if q)).strip()
    answer = " ".join(a for a in answers if a).strip()

    if not question or not answer:
        return None  # нет полноценной пары — пропускаем

    return {
        "ticket_id": ticket_id,
        "question": question,
        "answer": answer,
        "date": ticket.get("date_created", ""),
    }


def main():
    rows = []
    page = 1

    while True:
        data = get_tickets(page)
        tickets = data.get("data", {})
        if not tickets:
            break

        for ticket in tickets.values():
            qa = build_qa(ticket)
            if qa:
                rows.append(qa)
                print(f"  ✓ тикет {qa['ticket_id']}: {qa['question'][:60]}...")
            time.sleep(REQUEST_PAUSE)

        pagination = data.get("pagination", {})
        total_pages = pagination.get("total_pages", 1)
        print(f"страница {page}/{total_pages} обработана")
        if page >= total_pages:
            break
        page += 1
        time.sleep(REQUEST_PAUSE)

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["ticket_id", "question", "answer", "date"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nГотово: {len(rows)} кейсов сохранено в {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

In [ ]:
"""
Импорт реальных кейсов из DaoDesk в обучающую базу бота.

Что делает:
  1. Берёт закрытые тикеты (постранично).
  2. Для каждого тикета вытаскивает посты (ответы).
  3. Разделяет сообщения клиента (вопрос) и сотрудников (ответ) по user_id.
  4. Чистит HTML и сохраняет пары "вопрос — ответ" в CSV.

Запуск:
  pip install requests beautifulsoup4
  python import_daodesk_cases.py
"""

import csv
import time

import requests
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# ─────────────────────────── НАСТРОЙКИ ───────────────────────────
DOMAIN = "https://ВАШ_ДОМЕН"            # ← адрес твоего DaoDesk, например https://company.daodesk.ai
EMAIL = "amyakaaaaadm@gmail.com"
API_KEY = "a5dc273b-5151-465e-aef5-057a9b10dbc9"

# id статусов "закрытый/решённый" — узнай через GET /status_list/ (в Postman).
# Если оставить пустую строку "" — возьмутся ВСЕ тикеты (в т.ч. без ответов).
CLOSED_STATUSES = "closed,resolved"

OUTPUT_CSV = "daodesk_cases.csv"
REQUEST_PAUSE = 0.4                      # пауза между запросами, чтобы не упереться в rate limit
# ──────────────────────────────────────────────────────────────────

BASE_URL = f"{DOMAIN}/api/v2"
auth = HTTPBasicAuth(EMAIL, API_KEY)


def strip_html(html: str) -> str:
    """Убирает HTML-теги, оставляет чистый текст."""
    return BeautifulSoup(html or "", "html.parser").get_text(separator=" ", strip=True)


def get_tickets(page: int) -> dict:
    params = {"page": page}
    if CLOSED_STATUSES:
        params["status_list"] = CLOSED_STATUSES
    resp = requests.get(f"{BASE_URL}/tickets/", params=params, auth=auth, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_posts(ticket_id: int) -> list:
    resp = requests.get(
        f"{BASE_URL}/tickets/{ticket_id}/posts/",
        params={"page": 1},
        auth=auth,
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json().get("data", [])


def build_qa(ticket: dict) -> dict | None:
    """Собирает пару вопрос-ответ из одного тикета."""
    ticket_id = ticket["id"]
    client_id = ticket.get("user_id")

    posts = get_posts(ticket_id)
    if not posts:
        return None

    # сортируем по id (по возрастанию ≈ по времени создания)
    posts.sort(key=lambda p: p.get("id", 0))

    questions = [strip_html(p.get("text", "")) for p in posts if p.get("user_id") == client_id]
    answers = [strip_html(p.get("text", "")) for p in posts if p.get("user_id") != client_id]

    # вопрос = тема тикета + сообщения клиента; ответ = ответы сотрудников
    question = (ticket.get("title", "") + " " + " ".join(q for q in questions if q)).strip()
    answer = " ".join(a for a in answers if a).strip()

    if not question or not answer:
        return None  # нет полноценной пары — пропускаем

    return {
        "ticket_id": ticket_id,
        "question": question,
        "answer": answer,
        "date": ticket.get("date_created", ""),
    }


def main():
    rows = []
    page = 1

    while True:
        data = get_tickets(page)
        tickets = data.get("data", {})
        if not tickets:
            break

        for ticket in tickets.values():
            qa = build_qa(ticket)
            if qa:
                rows.append(qa)
                print(f"  ✓ тикет {qa['ticket_id']}: {qa['question'][:60]}...")
            time.sleep(REQUEST_PAUSE)

        pagination = data.get("pagination", {})
        total_pages = pagination.get("total_pages", 1)
        print(f"страница {page}/{total_pages} обработана")
        if page >= total_pages:
            break
        page += 1
        time.sleep(REQUEST_PAUSE)

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["ticket_id", "question", "answer", "date"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nГотово: {len(rows)} кейсов сохранено в {OUTPUT_CSV}")


if __name__ == "__main__":
    main()